In [24]:
"""
Merger Retriever (LOTR -- Lord of the Retrievers) RAG Pipeline -- Production-Grade
====================================================================================
Architecture   : FIVE configurations covering the complete MergerRetriever design
                 space as established by LangChain official documentation and the
                 2024-2025 production literature:
                 (1) Baseline                        -- single dense FAISS retriever,
                                                        no merging, no post-processing
                 (2) LOTR Basic Merge                -- MergerRetriever combining
                                                        3 retrievers with 3 different
                                                        embedding models + search modes
                 (3) LOTR + EmbeddingsRedundantFilter -- merge then deduplicate via
                                                        cross-embedding cosine filter
                 (4) LOTR + EmbeddingsClusteringFilter -- merge then K-means cluster,
                                                        pick 1 centroid-closest per cluster
                 (5) LOTR + 4-Stage Pipeline + [BEST]  -- merge then:
                     LongContextReorder                  EmbeddingsRedundantFilter
                                                        + EmbeddingsClusteringFilter
                                                        + EmbeddingsFilter
                                                        + LongContextReorder (lost-in-middle fix)

Indexes        : THREE heterogeneous FAISS indexes, each indexed with a DIFFERENT
                 embedding model, enabling embedding-diversity benefits:
                     Index A: BAAI/bge-large-en-v1.5   (1024-dim, asymmetric bi-encoder)
                     Index B: sentence-transformers/all-MiniLM-L6-v2  (384-dim, symmetric)
                     Index C: sentence-transformers/multi-qa-MiniLM-L6-dot-v1 (384-dim, QA)
                 + BM25Plus sparse retriever (rank-bm25)
                 Filter Embeddings: BAAI/bge-large-en-v1.5 (used for post-merge filtering)

LLM            : Azure OpenAI  (ChatOllama -- GPT-4o)
Framework      : LangChain 0.2+ (MergerRetriever, ContextualCompressionRetriever,
                                  DocumentCompressorPipeline, EmbeddingsRedundantFilter,
                                  EmbeddingsClusteringFilter, LongContextReorder,
                                  EmbeddingsFilter)

Reference PDFs (open-access arXiv):
    1. "Attention Is All You Need" -- Vaswani et al., 2017
       https://arxiv.org/pdf/1706.03762.pdf
    2. "BERT: Pre-training of Deep Bidirectional Transformers" -- Devlin et al., 2018
       https://arxiv.org/pdf/1810.04805.pdf
    3. "RAG for Knowledge-Intensive NLP Tasks" -- Lewis et al., 2020
       https://arxiv.org/pdf/2005.11401.pdf

=============================================================================
What MergerRetriever (LOTR) Is and Why It Exists
=============================================================================
Lord of the Retrievers (LOTR), also known as MergerRetriever, takes a list of
retrievers as input and merges the results of their get_relevant_documents()
methods into a single list. The merged results will be a list of documents that
are relevant to the query and that have been ranked by the different retrievers.
The MergerRetriever class can be used to improve the accuracy of document retrieval
in a number of ways. First, it can combine the results of multiple retrievers,
which can help to reduce the risk of bias in the results. Second, it can rank the
results of the different retrievers, which can help to ensure that the most relevant
documents are returned first.

MergerRetriever vs EnsembleRetriever -- The Production Distinction:
    EnsembleRetriever performs RECIPROCAL RANK FUSION (RRF) across its retrievers.
    Each document receives a combined RRF score; the output is a single re-scored,
    globally ranked list. The ranking is determined by the RRF formula, not raw retrieval order.

    MergerRetriever performs a ROUND-ROBIN INTERLEAVE of its retrievers' output lists.
    The MergerRetriever functions by combining the findings from various retrieval
    sources in a sequential, round-robin manner. It starts by collecting relevant
    documents identified by each retriever and then amalgamates these into a single,
    cohesive list. There is NO score-based re-ranking: document 1 from retriever A,
    then document 1 from retriever B, then document 2 from A, document 2 from B, etc.

    WHEN TO USE MergerRetriever over EnsembleRetriever:
        - You want to preserve per-retriever ordering as a signal for downstream
          DocumentCompressorPipeline stages.
        - You are combining retrievers with INCOMPARABLE scoring systems (e.g., a BM25
          retriever whose scores are TF-IDF weights alongside a dense retriever whose
          scores are cosine similarities). RRF (EnsembleRetriever) can still work here,
          but MergerRetriever makes the interleaving transparent and inspectable.
        - You have MORE THAN 3 retrievers and want fine-grained control over which
          retriever's documents appear first in the merged list.
        - You are building a pipeline where post-merge filtering (EmbeddingsRedundantFilter,
          EmbeddingsClusteringFilter) is the primary deduplication mechanism, not RRF.
        - The retrievers are backed by DIFFERENT VECTOR STORES or DIFFERENT DOMAINS
          (internal database + web search + BM25): MergerRetriever is specifically
          designed to merge outputs from entirely different retrieval architectures.

=============================================================================
The Embedding Diversity Strategy: Why Three Different Models
=============================================================================
This pipeline indexes the SAME corpus into THREE separate FAISS indexes, each
built with a DIFFERENT embedding model. This embedding diversity strategy is the
core architectural innovation of the LOTR pattern.

    INDEX A -- BAAI/bge-large-en-v1.5 (1024-dim, asymmetric):
        Trained with asymmetric contrastive learning: queries and documents can have
        different embedding norms. Optimized for retrieval tasks where the query is
        short and the document is long. BGE-large is the strongest general-purpose
        embedding model for retrieval in the 1B-param-trained-on-MSMARCO category.
        Biased toward: semantic similarity, domain-specific technical vocabulary.

    INDEX B -- all-MiniLM-L6-v2 (384-dim, symmetric):
        Trained with symmetric contrastive learning: similar texts map to similar
        embeddings regardless of whether they are queries or documents.
        Biased toward: general semantic similarity, broad topic matching.
        Captures different aspects of relevance than BGE: a document that scores
        highly on all-MiniLM may not score highly on BGE and vice versa.

    INDEX C -- multi-qa-MiniLM-L6-dot-v1 (384-dim, QA-specialized):
        Fine-tuned specifically on question-answer pairs. The dot-product metric
        (L2 normalization disabled) means similarity is magnitude-sensitive.
        Biased toward: query-answer structural alignment. Documents phrased in an
        "answer to a question" style score highest here.

Why this diversity matters:
    No single embedding model captures all aspects of semantic relevance.
    Using multiples embeddings in diff steps could help reduce biases.
    A document retrieved by ALL THREE models has been judged relevant by three
    independent semantic representations -- far stronger signal than one model alone.
    A document retrieved by only ONE model may be a model-specific false positive.
    The EmbeddingsRedundantFilter applied post-merge eliminates duplicates across
    all three index outputs using a FOURTH embedding for the filtering comparison,
    ensuring the redundancy check itself is not biased by any single retrieval model.

=============================================================================
LongContextReorder: The Lost-in-the-Middle Fix
=============================================================================
There is a substantial performance degradation when you include 10+ retrieved
documents. When models must access relevant information in the middle of long
contexts, they tend to ignore the provided documents. LongContextReorder
reorders documents to mitigate this problem.

Algorithm: Documents are sorted by relevance score and then REORDERED such that
the most relevant documents are placed at the BEGINNING and END of the context,
with the least relevant documents in the MIDDLE:
    [doc1_best, doc3_good, doc5_ok, doc6_ok, doc4_good, doc2_best]
    Position 1:  most relevant
    Position 2:  3rd most relevant
    Position 3:  5th most relevant (middle -- least important)
    Position 4:  6th most relevant (middle -- least important)
    Position 5:  4th most relevant
    Position 6:  2nd most relevant  (last position -- high attention)

This bimodal placement ensures that both LLM attention modes (beginning-anchored
and recency-biased) receive the highest-quality context rather than having the
most important documents stranded in the middle.


"""

'\nMerger Retriever (LOTR -- Lord of the Retrievers) RAG Pipeline -- Production-Grade\n====================================================================================\nArchitecture   : FIVE configurations covering the complete MergerRetriever design\n                 space as established by LangChain official documentation and the\n                 2024-2025 production literature:\n                 (1) Baseline                        -- single dense FAISS retriever,\n                                                        no merging, no post-processing\n                 (2) LOTR Basic Merge                -- MergerRetriever combining\n                                                        3 retrievers with 3 different\n                                                        embedding models + search modes\n                 (3) LOTR + EmbeddingsRedundantFilter -- merge then deduplicate via\n                                                        cross-embedding cosine filter\n    

In [25]:
import os
import sys
import time
import pickle
import logging
import urllib.request
from dataclasses import dataclass, field
from pathlib import Path
from typing import List, Dict, Tuple, Optional, Any

import numpy as np

In [26]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.retrievers import BM25Retriever
from langchain_community.document_transformers import (
    EmbeddingsRedundantFilter,
    EmbeddingsClusteringFilter,
    LongContextReorder,
)
from langchain_classic.retrievers import (
    MergerRetriever,
    ContextualCompressionRetriever,
    EnsembleRetriever,
)
from langchain_classic.retrievers.document_compressors import (
    EmbeddingsFilter, LLMChainExtractor, LLMChainFilter, LLMListwiseRerank, DocumentCompressorPipeline
)
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import BaseOutputParser, StrOutputParser


In [27]:
# ===========================================================================
# LOGGING
# ===========================================================================
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(name)s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    handlers=[logging.StreamHandler(sys.stdout)],
)
logger = logging.getLogger("lotr_rag")


In [28]:

# ===========================================================================
# SECTION 1 -- CONFIGURATION
# ===========================================================================

class LOTRConfig:
    """
    Centralized configuration for the LOTR (MergerRetriever) RAG pipeline.

    EMBEDDING MODELS:
        Three embedding models are deliberately chosen for maximum diversity:
            BGE_LARGE:    state-of-the-art asymmetric bi-encoder (retrieval-specialized)
            MINILM_ALL:   symmetric general-purpose encoder (broad semantic coverage)
            MINILM_QA:    QA-fine-tuned dot-product encoder (query-answer alignment)
        FILTER_EMBEDDINGS uses BGE_LARGE for all post-merge filtering operations.
        Using a SEPARATE embedding model for filtering than for retrieval is the
        recommended practice: Using multiples embeddings in diff steps could help
        reduce biases from any single model's perspective.

    MERGER_K (5 per retriever):
        Each retriever returns K=5 documents. With 4 retrievers (3 dense + BM25),
        the MergerRetriever round-robin merge produces up to 20 candidates.
        After EmbeddingsRedundantFilter (threshold=0.95): expect ~8-12 unique docs.
        After EmbeddingsClusteringFilter (num_clusters=4, num_closest=1): expect exactly 4 docs.
        After LongContextReorder: same 4 docs, reordered for lost-in-the-middle mitigation.

    CLUSTERING_NUM_CLUSTERS (4):
        EmbeddingsClusteringFilter performs K-means clustering on the merged document
        embeddings. K=4 clusters means the final context contains exactly 4 documents,
        each representing a different semantic "center of meaning" from the merged set.
        This ensures TOPICAL DIVERSITY in the final context:
            Cluster 1: documents about attention mechanism computation
            Cluster 2: documents about training procedure
            Cluster 3: documents about architecture components
            Cluster 4: documents about evaluation results
        Without clustering, the top-4 documents by raw score might all be from cluster 1,
        leaving the LLM with redundant coverage of one topic and gaps in others.

    CLUSTERING_NUM_CLOSEST (1):
        Returns 1 document per cluster center (the centroid-nearest document).
        Setting num_closest=2 would return 2 documents per cluster = 8 total
        at 4 clusters. In production, num_closest=1 for tight context budget control;
        num_closest=2 for wider coverage at the cost of context length.

    REDUNDANCY_THRESHOLD (0.95):
        Inherited from prior pipelines. Two documents with cosine similarity > 0.95
        (via the filter embedding model) are near-identical. Drop the second occurrence.
        The threshold is applied using the FILTER_EMBEDDINGS model (BGE-large), not the
        same model that produced the retrieval result. This cross-model redundancy check
        is more robust than same-model redundancy: a document that appears unique from
        the perspective of all-MiniLM may still be identical in BGE-large space.

    EMBEDDINGS_FILTER_THRESHOLD (0.72):
        Applied in Config 5's pipeline after redundancy removal. Drops documents
        with cosine similarity to the query below 0.72 (BGE-large-calibrated threshold).
    """

    # --- Dataset -----------------------------------------------------------
    PDF_SOURCES: List[Tuple[str, str]] = [
        ("attention_is_all_you_need",     "https://arxiv.org/pdf/1706.03762.pdf"),
        ("bert_pretraining_transformers",  "https://arxiv.org/pdf/1810.04805.pdf"),
        ("rag_knowledge_intensive_nlp",   "https://arxiv.org/pdf/2005.11401.pdf"),
    ]
    PDF_DOWNLOAD_DIR: str = "./pdfs"

    # --- FAISS Index Directories -----------------------------------------
    FAISS_BGE_DIR:     str = "./faiss_lotr_bge"
    FAISS_MINILM_DIR:  str = "./faiss_lotr_minilm"
    FAISS_QA_DIR:      str = "./faiss_lotr_qa"

    # --- BM25 ------------------------------------------------------------
    BM25_PERSIST_DIR: str  = "./bm25_lotr"
    BM25_PARAMS:      Dict = {"k1": 1.5, "b": 0.75, "delta": 0.5}

    # --- Embedding Models -----------------------------------------------
    BGE_MODEL:           str = "BAAI/bge-large-en-v1.5"          # 1024-dim
    MINILM_ALL_MODEL:    str = "sentence-transformers/all-MiniLM-L6-v2"  # 384-dim
    MINILM_QA_MODEL:     str = "sentence-transformers/multi-qa-MiniLM-L6-dot-v1"  # 384-dim
    FILTER_EMBED_MODEL:  str = "BAAI/bge-large-en-v1.5"          # used for all filters
    BIENCODER_DEVICE:    str = "cpu"
    BGE_QUERY_INSTRUCTION: str = (
        "Represent this sentence for searching relevant passages: "
    )

    # --- Retriever K Parameters -----------------------------------------
    MERGER_K:    int = 5    # K per retriever inside MergerRetriever
    BM25_K:      int = 5    # BM25Plus retriever K
    FINAL_K:     int = 4    # target docs into LLM

    # --- Clustering Parameters ------------------------------------------
    CLUSTERING_NUM_CLUSTERS: int = 4    # K-means clusters
    CLUSTERING_NUM_CLOSEST:  int = 1    # docs per cluster
    CLUSTERING_RANDOM_STATE: int = 42   # reproducible K-means init

    # --- Filter Thresholds -----------------------------------------------
    REDUNDANCY_THRESHOLD:        float = 0.95   # EmbeddingsRedundantFilter
    EMBEDDINGS_FILTER_THRESHOLD: float = 0.72   # EmbeddingsFilter (Config 5)

    # --- LLM Temperature ------------------------------------------------
    LLM_ANSWER_TEMPERATURE: float = 0.0

    # --- Text Splitting (index-time) ------------------------------------
    CHUNK_SIZE:    int = 450
    CHUNK_OVERLAP: int = 60

    # --- Azure OpenAI LLM -----------------------------------------------
    OLLAMA_BASE_URL: str = os.environ.get("OLLAMA_BASE_URL", "http://localhost:11434")
    OLLAMA_MODEL:    str = os.environ.get("OLLAMA_MODEL",    "qwen2.5-coder:7b")
    LLM_MAX_TOKENS: int            = 1024

    # --- RAG Answer Prompt -----------------------------------------------
    RAG_PROMPT: str = """You are a precise technical research assistant.
Answer the question using ONLY the information in the context below.
If the answer is not present, respond:
"The provided documents do not contain sufficient information to answer this question."

Context (retrieved via multi-retriever LOTR merge):
{context}

Question: {question}

Precise technical answer:"""


In [29]:


# ===========================================================================
# SECTION 2 -- LOTR TRACE DATA CLASS
# ===========================================================================

@dataclass
class LOTRTrace:
    """
    Records the complete execution trace of a LOTR MergerRetriever run.

    pre_filter_count:  total documents returned by the MergerRetriever before
        any DocumentCompressorPipeline post-processing. This is the raw merge count:
        n_retrievers * K_per_retriever (minus any retriever that returned fewer than K).

    post_filter_count: documents remaining after the DocumentCompressorPipeline.
        Deduplication, clustering, and relevance filtering reduce this significantly.

    retriever_contributions: dict of {retriever_name: List[doc_preview]} showing
        which retriever contributed which documents to the merged set. Enables
        per-retriever observability: which retrievers are pulling their weight?

    reorder_applied: True if LongContextReorder was applied in the pipeline.
        When True, the final document order is NOT by relevance score but by the
        bimodal lost-in-the-middle mitigation ordering.
    """
    question:               str
    strategy:               str
    pre_filter_count:       int            = 0
    post_filter_count:      int            = 0
    final_docs:             List[Document] = field(default_factory=list)
    retriever_contributions: Dict[str, int] = field(default_factory=dict)
    reorder_applied:        bool           = False
    final_answer:           str            = ""
    retrieval_ms:           float          = 0.0
    filter_ms:              float          = 0.0
    generation_ms:          float          = 0.0
    total_ms:               float          = 0.0

    def print_trace(self) -> None:
        print(f"\n{'='*74}")
        print(f"TRACE: {self.strategy}")
        print(f"Query: {self.question[:80]}")
        print(f"{'='*74}")
        print(f"\n  Merged (pre-filter): {self.pre_filter_count} docs")
        print(f"  After pipeline:      {self.post_filter_count} docs")
        print(f"  Final to LLM:        {len(self.final_docs)} docs")
        print(f"  LongContextReorder:  {'applied' if self.reorder_applied else 'not applied'}")

        if self.retriever_contributions:
            print(f"\n  Retriever contributions:")
            for name, count in self.retriever_contributions.items():
                print(f"    {name:<40}: {count} docs")

        print(f"\n  Final docs:")
        for i, doc in enumerate(self.final_docs, 1):
            paper = doc.metadata.get("paper_name", "?")[:30]
            page  = doc.metadata.get("page", "?")
            chars = len(doc.page_content)
            print(f"    [{i}] {paper} p{page}  ({chars} chars)")

        print(f"\n  Latency: retrieval={self.retrieval_ms:.0f}ms  "
              f"filter={self.filter_ms:.0f}ms  "
              f"gen={self.generation_ms:.0f}ms  "
              f"total={self.total_ms:.0f}ms")
        print(f"\n  ANSWER:\n  {self.final_answer[:450]}")
        print("=" * 74 + "\n")



In [30]:

# ===========================================================================
# SECTION 3 -- PDF DOWNLOADER AND CHUNKER
# ===========================================================================

def download_pdfs(config: LOTRConfig) -> List[Path]:
    """Download research PDFs with file-existence caching."""
    dl_dir = Path(config.PDF_DOWNLOAD_DIR)
    dl_dir.mkdir(parents=True, exist_ok=True)
    paths: List[Path] = []
    for name, url in config.PDF_SOURCES:
        dest = dl_dir / f"{name}.pdf"
        if dest.exists() and dest.stat().st_size > 10_000:
            logger.info("Cached: %s  (%.1f KB)", dest.name, dest.stat().st_size / 1024)
            paths.append(dest)
            continue
        logger.info("Downloading: %s", url)
        try:
            req = urllib.request.Request(
                url, headers={"User-Agent": "Mozilla/5.0 (RAG-LOTR-Pipeline/1.0)"}
            )
            with urllib.request.urlopen(req, timeout=60) as resp:
                data = resp.read()
            if len(data) < 10_000:
                raise RuntimeError(f"File too small: {url}")
            dest.write_bytes(data)
            logger.info("Saved: %s  (%.1f KB)", dest.name, len(data) / 1024)
            paths.append(dest)
            time.sleep(1.0)
        except Exception as exc:
            raise RuntimeError(
                f"Cannot download '{url}'. Place manually at '{dest}'."
            ) from exc
    return paths


def load_and_chunk_documents(
    pdf_paths: List[Path],
    config: LOTRConfig,
) -> List[Document]:
    """Load PDFs and split into chunks with paper_name and page metadata."""
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=config.CHUNK_SIZE,
        chunk_overlap=config.CHUNK_OVERLAP,
        separators=["\n\n", "\n", ". ", "! ", "? ", " ", ""],
        add_start_index=True,
    )
    all_chunks: List[Document] = []
    for pdf_path in pdf_paths:
        paper_name = pdf_path.stem.replace("_", " ").title()
        try:
            pages = PyPDFLoader(str(pdf_path)).load()
        except Exception as exc:
            raise RuntimeError(f"Failed to load '{pdf_path}': {exc}") from exc
        for page in pages:
            page.metadata.update({"source": pdf_path.name, "paper_name": paper_name})
        chunks = splitter.split_documents(pages)
        logger.info("  %s -> %d pages -> %d chunks", pdf_path.name, len(pages), len(chunks))
        all_chunks.extend(chunks)
    logger.info("Total chunks: %d", len(all_chunks))
    return all_chunks


In [54]:


# ===========================================================================
# SECTION 4 -- INDEX AND MODEL BUILDERS
# ===========================================================================

def build_embedding_model(
    model_name: str,
    device:     str = "cpu",
    query_instruction: str = "",
    normalize:  bool = True,
) -> HuggingFaceEmbeddings:
    """Build a HuggingFace embedding model with configurable parameters."""
    logger.info("Loading embedding: %s", model_name)
    kwargs: Dict = {"model_name": model_name, "model_kwargs": {"device": device}}
    if normalize:
        kwargs["encode_kwargs"] = {"normalize_embeddings": True}

    # Compatibility: some langchain-community versions reject query_instruction.
    supports_query_instruction = "query_instruction" in getattr(HuggingFaceEmbeddings, "model_fields", {})
    if query_instruction and supports_query_instruction:
        kwargs["query_instruction"] = query_instruction
    elif query_instruction:
        logger.warning(
            "query_instruction is not supported by this HuggingFaceEmbeddings version; ignoring it."
        )

    return HuggingFaceEmbeddings(**kwargs)


def build_faiss_index(
    chunks:     List[Document],
    embeddings: HuggingFaceEmbeddings,
    persist_dir: str,
) -> FAISS:
    """Build or load a FAISS index from chunks using the given embedding model."""
    faiss_file = Path(persist_dir) / "index.faiss"
    if faiss_file.exists():
        logger.info("Loading FAISS from '%s' ...", persist_dir)
        try:
            vs = FAISS.load_local(
                persist_dir, embeddings,
                allow_dangerous_deserialization=True,
            )
            logger.info("  Loaded. Vectors: %d", vs.index.ntotal)
            return vs
        except Exception as exc:
            logger.warning("FAISS load failed (%s), rebuilding ...", exc)
    logger.info("Building FAISS in '%s' from %d chunks ...", persist_dir, len(chunks))
    t0 = time.perf_counter()
    vs = FAISS.from_documents(chunks, embeddings)
    logger.info("  Built. Vectors: %d  (%.2fs)", vs.index.ntotal, time.perf_counter() - t0)
    Path(persist_dir).mkdir(parents=True, exist_ok=True)
    vs.save_local(persist_dir)
    return vs


def bm25_preprocess_text(text: str) -> List[str]:
    """Pickle-safe BM25 tokenization function."""
    return text.lower().split()


def build_bm25_index(chunks: List[Document], config: LOTRConfig) -> BM25Retriever:
    """Build or load BM25 retriever with pickle persistence and version-safe params."""
    cache = Path(config.BM25_PERSIST_DIR) / "bm25plus.pkl"
    if cache.exists():
        logger.info("Loading BM25 from '%s' ...", cache)
        try:
            with open(cache, "rb") as f:
                state = pickle.load(f)
            return BM25Retriever(
                vectorizer=state["vectorizer"], docs=state["docs"],
                k=state.get("k", config.BM25_K),
                preprocess_func=state.get("preprocess_func", bm25_preprocess_text),
            )
        except Exception as exc:
            logger.warning("BM25 cache load failed (%s); rebuilding ...", exc)
            try:
                cache.unlink()
            except OSError:
                pass

    logger.info("Building BM25Plus from %d chunks ...", len(chunks))
    bm25_params = dict(config.BM25_PARAMS)

    try:
        ret = BM25Retriever.from_documents(
            chunks, bm25_variant="plus",
            bm25_params=bm25_params,
            preprocess_func=bm25_preprocess_text,
        )
    except TypeError as exc:
        # Compatibility: some langchain-community versions instantiate BM25Okapi
        # regardless of bm25_variant and reject BM25Plus-only params (e.g., delta).
        if "delta" not in str(exc):
            raise
        logger.warning(
            "BM25Plus parameters unsupported by this BM25Retriever version; falling back to BM25Okapi-safe params."
        )
        safe_params = {k: v for k, v in bm25_params.items() if k in {"k1", "b", "epsilon"}}
        ret = BM25Retriever.from_documents(
            chunks,
            bm25_params=safe_params,
            preprocess_func=bm25_preprocess_text,
        )

    ret.k = config.BM25_K
    cache.parent.mkdir(parents=True, exist_ok=True)
    with open(cache, "wb") as f:
        pickle.dump({"vectorizer": ret.vectorizer, "docs": ret.docs, "k": ret.k},
                    f, protocol=pickle.HIGHEST_PROTOCOL)
    return ret


def build_ollama_llm(config: LOTRConfig) -> ChatOllama:
    """Connect to local Ollama and return a ChatOllama instance."""
    import urllib.request, urllib.error
    base_url = getattr(config, 'OLLAMA_BASE_URL', 'http://localhost:11434')
    model    = getattr(config, 'OLLAMA_MODEL',    'qwen2.5-coder:7b')
    try:
        urllib.request.urlopen(base_url, timeout=3)
    except Exception as exc:
        raise RuntimeError(
            f"Ollama not reachable at {base_url}. Start Ollama and run: "
            f"ollama pull qwen2.5-coder:7b"
        ) from exc
    return ChatOllama(
        base_url=base_url,
        model=model,
        temperature=getattr(config, 'LLM_TEMPERATURE', 0.7),
        num_predict=getattr(config, 'LLM_MAX_TOKENS', 512),
    )

In [32]:
# ===========================================================================
# SECTION 5 -- SHARED UTILITIES
# ===========================================================================

def format_docs(docs: List[Document]) -> str:
    """Format documents into annotated context block for LLM answer generation."""
    parts = []
    for i, doc in enumerate(docs, 1):
        paper = doc.metadata.get("paper_name", "Unknown")
        page  = doc.metadata.get("page", "?")
        parts.append(
            f"[{i} | {paper} | p{page}]\n{doc.page_content.strip()}"
        )
    return ("\n\n" + "-" * 60 + "\n\n").join(parts)


def generate_answer(
    question: str,
    docs:     List[Document],
    llm:      ChatOllama,
    config:   LOTRConfig,
) -> Tuple[str, float]:
    """Generate answer from final documents. Returns (answer, latency_ms)."""
    context = format_docs(docs)
    prompt  = ChatPromptTemplate.from_template(config.RAG_PROMPT)
    t0      = time.perf_counter()
    answer  = (prompt | llm | StrOutputParser()).invoke(
        {"context": context, "question": question}
    )
    return answer, (time.perf_counter() - t0) * 1000


def build_per_retriever_snapshot(
    question: str,
    retrievers: Dict[str, Any],
) -> Dict[str, int]:
    """
    Capture how many documents each named retriever returns for observability.

    In production, this per-retriever count enables diagnosing retrieval balance:
    if BM25 returns 0 docs on a query while all dense retrievers return K,
    the query is likely purely semantic (BM25 cannot match).
    If one dense retriever consistently returns fewer than K,
    investigate the embedding model's coverage of that query category.
    """
    counts = {}
    for name, retriever in retrievers.items():
        try:
            docs = retriever.invoke(question)
            counts[name] = len(docs)
        except Exception:
            counts[name] = 0
    return counts


In [33]:

# ===========================================================================
# SECTION 6 -- FIVE LOTR CONFIGURATIONS
# ===========================================================================

# ---------------------------------------------------------------------------
# CONFIG 1: Baseline -- single BGE-large FAISS retriever
# ---------------------------------------------------------------------------

def run_config1_baseline(
    question:          str,
    faiss_bge:         FAISS,
    llm:               ChatOllama,
    config:            LOTRConfig,
) -> LOTRTrace:
    """
    Configuration 1: Single-retriever baseline.

    FAISS + BGE-large only. Top-FINAL_K by cosine similarity.
    No merging, no deduplication, no reordering. Control condition.
    """
    trace = LOTRTrace(question=question, strategy="Config1_Baseline_SingleRetriever")
    t0    = time.perf_counter()

    retriever = faiss_bge.as_retriever(
        search_type="similarity",
        search_kwargs={"k": config.FINAL_K},
    )
    t_ret  = time.perf_counter()
    docs   = retriever.invoke(question)
    trace.retrieval_ms    = (time.perf_counter() - t_ret) * 1000
    trace.pre_filter_count  = len(docs)
    trace.post_filter_count = len(docs)
    trace.final_docs        = docs[: config.FINAL_K]
    trace.retriever_contributions = {"FAISS_BGE": len(docs)}

    answer, trace.generation_ms = generate_answer(question, trace.final_docs, llm, config)
    trace.final_answer = answer
    trace.total_ms     = (time.perf_counter() - t0) * 1000
    return trace


In [34]:

# ---------------------------------------------------------------------------
# CONFIG 2: LOTR Basic Merge -- 4 retrievers, round-robin interleave
# ---------------------------------------------------------------------------

def run_config2_lotr_basic(
    question:   str,
    faiss_bge:  FAISS,
    faiss_mini: FAISS,
    faiss_qa:   FAISS,
    bm25:       BM25Retriever,
    llm:        ChatOllama,
    config:     LOTRConfig,
) -> LOTRTrace:
    """
    Configuration 2: LOTR Basic MergerRetriever -- 4 retrievers, no post-processing.

    MergerRetriever merges results from:
        - FAISS + BGE-large (similarity search, K=5)
        - FAISS + all-MiniLM-L6-v2 (MMR search, K=5, lambda_mult=0.7)
        - FAISS + multi-qa-MiniLM-L6-dot (similarity search, K=5)
        - BM25Plus sparse retriever (K=5)

    The round-robin interleave order:
        Position 1:  BGE-large rank-1 doc
        Position 2:  MiniLM rank-1 doc
        Position 3:  QA-MiniLM rank-1 doc
        Position 4:  BM25 rank-1 doc
        Position 5:  BGE-large rank-2 doc
        Position 6:  MiniLM rank-2 doc
        ...

    No deduplication in this config. The raw merged list may contain duplicates
    when two different embedding models retrieve the same chunk. This config shows
    the raw LOTR output for comparison with filtered configs.

    MMR for MiniLM retriever (lambda_mult=0.7):
        Maximum Marginal Relevance selects documents that are both relevant to the
        query AND diverse from each other. Applied to the MiniLM retriever specifically
        because it tends to return highly similar documents (symmetric encoder with
        less domain specificity than BGE). MMR diversifies within the MiniLM results
        before they enter the merger.

    Args:
        question  : User query.
        faiss_bge : FAISS index with BGE-large embeddings.
        faiss_mini: FAISS index with all-MiniLM-L6-v2 embeddings.
        faiss_qa  : FAISS index with multi-qa-MiniLM-L6-dot-v1 embeddings.
        bm25      : BM25Plus retriever.
        llm       : ChatOllama for answer generation.
        config    : LOTRConfig.

    Returns:
        LOTRTrace with raw merged document counts per retriever.
    """
    trace = LOTRTrace(question=question, strategy="Config2_LOTR_BasicMerge")
    t0    = time.perf_counter()

    # Retriever A: BGE-large, cosine similarity
    ret_bge = faiss_bge.as_retriever(
        search_type="similarity",
        search_kwargs={"k": config.MERGER_K},
    )
    # Retriever B: all-MiniLM, MMR for intra-model diversity
    ret_mini = faiss_mini.as_retriever(
        search_type="mmr",
        search_kwargs={
            "k": config.MERGER_K,
            "fetch_k": config.MERGER_K * 3,
            "lambda_mult": 0.7,
        },
    )
    # Retriever C: multi-qa-MiniLM, cosine similarity, QA-specialized
    ret_qa = faiss_qa.as_retriever(
        search_type="similarity",
        search_kwargs={"k": config.MERGER_K},
    )
    # Retriever D: BM25Plus sparse
    bm25_r = BM25Retriever(
        vectorizer=bm25.vectorizer, docs=bm25.docs,
        k=config.BM25_K, preprocess_func=bm25.preprocess_func,
    )

    # MergerRetriever: round-robin interleave
    lotr = MergerRetriever(retrievers=[ret_bge, ret_mini, ret_qa, bm25_r])

    # Capture per-retriever contributions
    trace.retriever_contributions = build_per_retriever_snapshot(question, {
        "FAISS_BGE_large": ret_bge,
        "FAISS_MiniLM_MMR": ret_mini,
        "FAISS_QA_MiniLM": ret_qa,
        "BM25Plus": bm25_r,
    })

    t_ret  = time.perf_counter()
    merged_docs = lotr.invoke(question)
    trace.retrieval_ms    = (time.perf_counter() - t_ret) * 1000
    trace.pre_filter_count  = len(merged_docs)
    trace.post_filter_count = len(merged_docs)
    trace.final_docs        = merged_docs[: config.FINAL_K]

    logger.info("Config2 LOTR Basic: merged %d docs -> top-%d to LLM",
                len(merged_docs), len(trace.final_docs))

    answer, trace.generation_ms = generate_answer(question, trace.final_docs, llm, config)
    trace.final_answer = answer
    trace.total_ms     = (time.perf_counter() - t0) * 1000
    return trace

In [35]:


# ---------------------------------------------------------------------------
# CONFIG 3: LOTR + EmbeddingsRedundantFilter
# ---------------------------------------------------------------------------

def run_config3_lotr_redundant_filter(
    question:        str,
    faiss_bge:       FAISS,
    faiss_mini:      FAISS,
    faiss_qa:        FAISS,
    bm25:            BM25Retriever,
    filter_embeddings: HuggingFaceEmbeddings,
    llm:             ChatOllama,
    config:          LOTRConfig,
) -> LOTRTrace:
    """
    Configuration 3: LOTR + EmbeddingsRedundantFilter.

    Extends Config 2 by adding an EmbeddingsRedundantFilter as a
    DocumentCompressorPipeline inside a ContextualCompressionRetriever.
    The LOTR merger is the base retriever; the filter is the compressor.

    EmbeddingsRedundantFilter mechanics:
        - Embeds ALL merged documents using the FILTER_EMBEDDINGS model (BGE-large).
        - Iterates through the merged list in order (maintaining round-robin order).
        - For each document D_i, computes cosine similarity to all already-accepted
          documents in the output set.
        - If max(cosine(D_i, accepted)) > REDUNDANCY_THRESHOLD=0.95: DROP D_i.
        - If max(cosine(D_i, accepted)) <= 0.95: KEEP D_i, add to accepted set.

    Why a THIRD embedding model for the filter:
        The filter uses FILTER_EMBEDDINGS (BGE-large) even though the retrieval
        used BGE-large, MiniLM, and QA-MiniLM. This is intentional:
        We use a SEPARATE embedding model for redundancy detection to avoid the
        situation where two documents that are similar in BGE-large space were
        retrieved as diverse by QA-MiniLM (because they are similar to the query
        from different semantic angles). Using BGE-large for the filter provides
        a consistent, high-quality redundancy measurement that is independent of
        the retrieval embedding model choice.

    Args:
        question          : User query.
        faiss_bge/mini/qa : Three FAISS indexes.
        bm25              : BM25Plus retriever.
        filter_embeddings : BGE-large used for redundancy comparison.
        llm               : ChatOllama.
        config            : LOTRConfig.

    Returns:
        LOTRTrace with pre/post-filter counts showing deduplication rate.
    """
    trace = LOTRTrace(question=question, strategy="Config3_LOTR_RedundantFilter")
    t0    = time.perf_counter()

    ret_bge  = faiss_bge.as_retriever(search_type="similarity",
                                       search_kwargs={"k": config.MERGER_K})
    ret_mini = faiss_mini.as_retriever(search_type="mmr",
                                        search_kwargs={"k": config.MERGER_K,
                                                       "fetch_k": config.MERGER_K*3,
                                                       "lambda_mult": 0.7})
    ret_qa   = faiss_qa.as_retriever(search_type="similarity",
                                      search_kwargs={"k": config.MERGER_K})
    bm25_r   = BM25Retriever(vectorizer=bm25.vectorizer, docs=bm25.docs,
                              k=config.BM25_K, preprocess_func=bm25.preprocess_func)

    lotr = MergerRetriever(retrievers=[ret_bge, ret_mini, ret_qa, bm25_r])

    trace.retriever_contributions = build_per_retriever_snapshot(question, {
        "FAISS_BGE_large": ret_bge, "FAISS_MiniLM_MMR": ret_mini,
        "FAISS_QA_MiniLM": ret_qa, "BM25Plus": bm25_r,
    })

    # EmbeddingsRedundantFilter: drops near-duplicates using FILTER_EMBEDDINGS
    redundant_filter = EmbeddingsRedundantFilter(
        embeddings=filter_embeddings,
        similarity_threshold=config.REDUNDANCY_THRESHOLD,
    )
    pipeline = DocumentCompressorPipeline(transformers=[redundant_filter])
    cc_retriever = ContextualCompressionRetriever(
        base_compressor=pipeline,
        base_retriever=lotr,
    )

    # Measure raw merge count separately for trace
    t_ret = time.perf_counter()
    raw_merged = lotr.invoke(question)
    trace.pre_filter_count = len(raw_merged)

    # Run the full pipeline (retrieval + filter)
    t_filter = time.perf_counter()
    filtered_docs = cc_retriever.invoke(question)
    trace.retrieval_ms = (time.perf_counter() - t_ret) * 1000
    trace.filter_ms    = (time.perf_counter() - t_filter) * 1000
    trace.post_filter_count = len(filtered_docs)
    trace.final_docs        = filtered_docs[: config.FINAL_K]

    logger.info(
        "Config3 LOTR+RedundantFilter: %d merged -> %d after dedup -> %d to LLM",
        trace.pre_filter_count, trace.post_filter_count, len(trace.final_docs),
    )

    answer, trace.generation_ms = generate_answer(question, trace.final_docs, llm, config)
    trace.final_answer = answer
    trace.total_ms     = (time.perf_counter() - t0) * 1000
    return trace



In [36]:

# ---------------------------------------------------------------------------
# CONFIG 4: LOTR + EmbeddingsClusteringFilter
# ---------------------------------------------------------------------------

def run_config4_lotr_clustering(
    question:        str,
    faiss_bge:       FAISS,
    faiss_mini:      FAISS,
    faiss_qa:        FAISS,
    bm25:            BM25Retriever,
    filter_embeddings: HuggingFaceEmbeddings,
    llm:             ChatOllama,
    config:          LOTRConfig,
) -> LOTRTrace:
    """
    Configuration 4: LOTR + EmbeddingsClusteringFilter.

    EmbeddingsClusteringFilter divides the merged document vectors into clusters
    or "centers" of meaning. Then it picks the closest document to that center
    for the final results. By default the result document will be ordered/grouped
    by clusters. If you want the final documents to be ordered by the original
    retriever scores you need to add sorted=True.

    K-means Clustering Mechanics:
        1. Embed all merged documents using FILTER_EMBEDDINGS (BGE-large).
        2. Run K-means with num_clusters=4, producing 4 cluster centroids.
        3. For each cluster, select num_closest=1 document closest to its centroid.
        4. Output: exactly num_clusters * num_closest = 4 documents.

    Why clustering is superior to simple top-K selection for diverse queries:
        Simple top-K by any ranking metric tends to return documents that are
        all highly similar to each other (they cover the most prominent topic in the
        query). If the query is "how do attention mechanisms and positional encodings
        work in the Transformer?" simple top-K may return 4 documents about attention
        and 0 about positional encoding.

        Clustering partitions the document space so that each cluster represents a
        DISTINCT SEMANTIC THEME present in the merged result set:
            Cluster 1 centroid: near attention mechanism documents
            Cluster 2 centroid: near positional encoding documents
            Cluster 3 centroid: near training procedure documents
            Cluster 4 centroid: near architecture overview documents

        The final 4 documents -- one per cluster -- cover all 4 themes, giving the
        LLM a topically diverse context rather than redundant single-theme coverage.

    sorted=True: Orders the output by the original retriever scores rather than
        by cluster assignment. This ensures documents are presented to the LLM
        in relevance order even after clustering selection, preserving the
        "most relevant first" signal in the context.

    Args:
        question          : User query.
        faiss_bge/mini/qa : Three FAISS indexes.
        bm25              : BM25Plus retriever.
        filter_embeddings : BGE-large for K-means embeddings.
        llm               : ChatOllama.
        config            : LOTRConfig.

    Returns:
        LOTRTrace with cluster-selected diverse final docs.
    """
    trace = LOTRTrace(question=question, strategy="Config4_LOTR_ClusteringFilter")
    t0    = time.perf_counter()

    ret_bge  = faiss_bge.as_retriever(search_type="similarity",
                                       search_kwargs={"k": config.MERGER_K})
    ret_mini = faiss_mini.as_retriever(search_type="mmr",
                                        search_kwargs={"k": config.MERGER_K,
                                                       "fetch_k": config.MERGER_K*3,
                                                       "lambda_mult": 0.7})
    ret_qa   = faiss_qa.as_retriever(search_type="similarity",
                                      search_kwargs={"k": config.MERGER_K})
    bm25_r   = BM25Retriever(vectorizer=bm25.vectorizer, docs=bm25.docs,
                              k=config.BM25_K, preprocess_func=bm25.preprocess_func)

    lotr = MergerRetriever(retrievers=[ret_bge, ret_mini, ret_qa, bm25_r])

    trace.retriever_contributions = build_per_retriever_snapshot(question, {
        "FAISS_BGE_large": ret_bge, "FAISS_MiniLM_MMR": ret_mini,
        "FAISS_QA_MiniLM": ret_qa, "BM25Plus": bm25_r,
    })

    # EmbeddingsClusteringFilter: K-means on merged docs, select centroid-nearest
    # sorted=True preserves retriever-order within clusters (relevance-ordered output)
    cluster_filter = EmbeddingsClusteringFilter(
        embeddings=filter_embeddings,
        num_clusters=config.CLUSTERING_NUM_CLUSTERS,
        num_closest=config.CLUSTERING_NUM_CLOSEST,
        sorted=True,
        random_state=config.CLUSTERING_RANDOM_STATE,
    )
    pipeline = DocumentCompressorPipeline(transformers=[cluster_filter])
    cc_retriever = ContextualCompressionRetriever(
        base_compressor=pipeline,
        base_retriever=lotr,
    )

    t_ret = time.perf_counter()
    raw_merged = lotr.invoke(question)
    trace.pre_filter_count = len(raw_merged)

    t_filter = time.perf_counter()
    clustered_docs = cc_retriever.invoke(question)
    trace.retrieval_ms      = (time.perf_counter() - t_ret) * 1000
    trace.filter_ms         = (time.perf_counter() - t_filter) * 1000
    trace.post_filter_count = len(clustered_docs)
    trace.final_docs        = clustered_docs[: config.FINAL_K]

    logger.info(
        "Config4 LOTR+Clustering: %d merged -> %d clustered (%d clusters x %d closest)",
        trace.pre_filter_count, trace.post_filter_count,
        config.CLUSTERING_NUM_CLUSTERS, config.CLUSTERING_NUM_CLOSEST,
    )

    answer, trace.generation_ms = generate_answer(question, trace.final_docs, llm, config)
    trace.final_answer = answer
    trace.total_ms     = (time.perf_counter() - t0) * 1000
    return trace



In [37]:

# ---------------------------------------------------------------------------
# CONFIG 5: LOTR + Full 4-Stage Pipeline + LongContextReorder [BEST]
# ---------------------------------------------------------------------------

def run_config5_lotr_full_pipeline(
    question:        str,
    faiss_bge:       FAISS,
    faiss_mini:      FAISS,
    faiss_qa:        FAISS,
    bm25:            BM25Retriever,
    filter_embeddings: HuggingFaceEmbeddings,
    llm:             ChatOllama,
    config:          LOTRConfig,
) -> LOTRTrace:
    """
    Configuration 5: LOTR + Full 4-Stage DocumentCompressorPipeline [BEST].

    This configuration chains all post-merge transformers in sequence inside
    a DocumentCompressorPipeline, combining:
        Stage 1: EmbeddingsRedundantFilter (threshold=0.95)
                 Drop near-duplicate documents across all 4 retrievers' outputs.
        Stage 2: EmbeddingsClusteringFilter (num_clusters=4, num_closest=1, sorted=True)
                 Select one centroid-nearest document per semantic cluster.
                 Guarantees topical diversity in the final context.
        Stage 3: EmbeddingsFilter (threshold=0.72)
                 Drop any surviving documents with cosine similarity < 0.72 to query.
                 Safety net: ensures ALL final documents are genuinely query-relevant.
        Stage 4: LongContextReorder
                 Reorder surviving documents so most/least relevant alternate
                 between positions 1-2 and N-1, N (bimodal placement) to mitigate
                 lost-in-the-middle degradation.

    The pipeline order matters:
        - Redundancy removal FIRST: eliminates duplicates before clustering,
          ensuring K-means cluster centers are not distorted by duplicate vectors.
        - Clustering SECOND: selects thematically diverse documents.
        - Relevance filter THIRD: removes any document that survived clustering
          but has low query-similarity (edge case: a cluster center may be
          the closest document to an off-topic cluster centroid).
        - Reordering LAST: applied after all filtering is complete, so the
          bimodal ordering is applied to the final document set only.

    LongContextReorder Algorithm (bimodal placement):
        Input: [doc1(0.91), doc2(0.87), doc3(0.83), doc4(0.79)] (by relevance)
        Output: [doc1, doc3, doc4, doc2] (alternating from ends inward)
        Position 1: most relevant (doc1) -- LLM attends here
        Position 2: 3rd most relevant (doc3) -- LLM reads early
        Position 3: 4th most relevant (doc4) -- middle (lower attention)
        Position 4: 2nd most relevant (doc2) -- LLM attends here (recency)
        Result: the 2nd-most-relevant document is NOT stranded in the middle.

    Args:
        question          : User query.
        faiss_bge/mini/qa : Three FAISS indexes.
        bm25              : BM25Plus retriever.
        filter_embeddings : BGE-large for all filter stages.
        llm               : ChatOllama.
        config            : LOTRConfig.

    Returns:
        LOTRTrace with full pipeline metrics and reorder_applied=True.
    """
    trace = LOTRTrace(
        question=question, strategy="Config5_LOTR_FullPipeline_LongContextReorder [BEST]"
    )
    t0 = time.perf_counter()

    ret_bge  = faiss_bge.as_retriever(search_type="similarity",
                                       search_kwargs={"k": config.MERGER_K})
    ret_mini = faiss_mini.as_retriever(search_type="mmr",
                                        search_kwargs={"k": config.MERGER_K,
                                                       "fetch_k": config.MERGER_K*3,
                                                       "lambda_mult": 0.7})
    ret_qa   = faiss_qa.as_retriever(search_type="similarity",
                                      search_kwargs={"k": config.MERGER_K})
    bm25_r   = BM25Retriever(vectorizer=bm25.vectorizer, docs=bm25.docs,
                              k=config.BM25_K, preprocess_func=bm25.preprocess_func)

    lotr = MergerRetriever(retrievers=[ret_bge, ret_mini, ret_qa, bm25_r])

    trace.retriever_contributions = build_per_retriever_snapshot(question, {
        "FAISS_BGE_large": ret_bge, "FAISS_MiniLM_MMR": ret_mini,
        "FAISS_QA_MiniLM": ret_qa, "BM25Plus": bm25_r,
    })

    # Stage 1: Deduplicate
    stage1_redundant = EmbeddingsRedundantFilter(
        embeddings=filter_embeddings,
        similarity_threshold=config.REDUNDANCY_THRESHOLD,
    )

    # Stage 2: Cluster -- 4 semantic centers, 1 doc per center, relevance-sorted
    stage2_cluster = EmbeddingsClusteringFilter(
        embeddings=filter_embeddings,
        num_clusters=config.CLUSTERING_NUM_CLUSTERS,
        num_closest=config.CLUSTERING_NUM_CLOSEST,
        sorted=True,
        random_state=config.CLUSTERING_RANDOM_STATE,
    )

    # Stage 3: Relevance gate -- drops any doc with cosine < 0.72 to query
    stage3_relevant = EmbeddingsFilter(
        embeddings=filter_embeddings,
        similarity_threshold=config.EMBEDDINGS_FILTER_THRESHOLD,
        k=None,
    )

    # Stage 4: LongContextReorder -- bimodal placement for lost-in-the-middle mitigation
    stage4_reorder = LongContextReorder()

    # Assemble the 4-stage DocumentCompressorPipeline
    full_pipeline = DocumentCompressorPipeline(
        transformers=[
            stage1_redundant,   # remove near-duplicates
            stage2_cluster,     # select thematically diverse set
            stage3_relevant,    # drop off-topic survivors
            stage4_reorder,     # reorder for LLM attention optimization
        ]
    )
    cc_retriever = ContextualCompressionRetriever(
        base_compressor=full_pipeline,
        base_retriever=lotr,
    )

    t_ret = time.perf_counter()
    raw_merged = lotr.invoke(question)
    trace.pre_filter_count = len(raw_merged)

    t_filter = time.perf_counter()
    final_docs = cc_retriever.invoke(question)
    trace.retrieval_ms      = (time.perf_counter() - t_ret) * 1000
    trace.filter_ms         = (time.perf_counter() - t_filter) * 1000
    trace.post_filter_count = len(final_docs)
    trace.final_docs        = final_docs[: config.FINAL_K]
    trace.reorder_applied   = True

    # Safety fallback: if pipeline over-filtered to 0, use top-K raw merged docs
    if not trace.final_docs:
        logger.warning(
            "Config5: full pipeline returned 0 docs (over-filtered). "
            "Falling back to raw top-%d.", config.FINAL_K
        )
        trace.final_docs = raw_merged[: config.FINAL_K]
        trace.reorder_applied = False

    logger.info(
        "Config5 LOTR+FullPipeline: %d merged -> %d after pipeline -> %d to LLM",
        trace.pre_filter_count, trace.post_filter_count, len(trace.final_docs),
    )

    answer, trace.generation_ms = generate_answer(question, trace.final_docs, llm, config)
    trace.final_answer = answer
    trace.total_ms     = (time.perf_counter() - t0) * 1000
    return trace



In [38]:

# ===========================================================================
# SECTION 7 -- COMPARATIVE RUNNER
# ===========================================================================

def run_all_configs(
    question:         str,
    faiss_bge:        FAISS,
    faiss_mini:       FAISS,
    faiss_qa:         FAISS,
    bm25:             BM25Retriever,
    filter_embeddings: HuggingFaceEmbeddings,
    llm:              ChatOllama,
    config:           LOTRConfig,
) -> Dict[str, LOTRTrace]:
    """Run all five LOTR configurations on a single question."""
    print("\n" + "#" * 78)
    print(f"QUERY: {question}")
    print("#" * 78)

    runners = {
        "Config1_Baseline": lambda q: run_config1_baseline(
            q, faiss_bge, llm, config),
        "Config2_LOTR_BasicMerge": lambda q: run_config2_lotr_basic(
            q, faiss_bge, faiss_mini, faiss_qa, bm25, llm, config),
        "Config3_LOTR_RedundantFilter": lambda q: run_config3_lotr_redundant_filter(
            q, faiss_bge, faiss_mini, faiss_qa, bm25, filter_embeddings, llm, config),
        "Config4_LOTR_ClusteringFilter": lambda q: run_config4_lotr_clustering(
            q, faiss_bge, faiss_mini, faiss_qa, bm25, filter_embeddings, llm, config),
        "Config5_LOTR_FullPipeline+LCR [BEST]": lambda q: run_config5_lotr_full_pipeline(
            q, faiss_bge, faiss_mini, faiss_qa, bm25, filter_embeddings, llm, config),
    }

    traces: Dict[str, LOTRTrace] = {}
    for label, fn in runners.items():
        print(f"\n{'='*62}\nRUNNING: {label}\n{'='*62}")
        try:
            trace = fn(question)
            trace.print_trace()
            traces[label] = trace
        except Exception as exc:
            logger.error("Config %s failed: %s", label, exc, exc_info=True)

    print("\n" + "=" * 78)
    print("LOTR MERGER RETRIEVER SUMMARY")
    print(f"{'Config':<50} {'Pre':>5} {'Post':>5} {'Final':>6} {'Total(ms)':>10}")
    print("-" * 78)
    for lbl, tr in traces.items():
        print(
            f"{lbl:<50} {tr.pre_filter_count:>5} "
            f"{tr.post_filter_count:>5} "
            f"{len(tr.final_docs):>6} "
            f"{tr.total_ms:>10.0f}"
        )
    print("=" * 78 + "\n")

    return traces

In [39]:
"""
    End-to-end LOTR MergerRetriever RAG pipeline orchestrator.

    Index build strategy:
        THREE separate FAISS indexes built from the SAME chunks but with
        DIFFERENT embedding models. Building 3 separate indexes from identical
        chunks enables embedding-diversity retrieval without requiring 3 separate
        corpora. Each index "sees" the same documents from a different semantic
        lens, ensuring that the LOTR merger captures all semantic perspectives.

    Memory considerations:
        BGE-large (1024-dim): ~1.3 GB model + index memory
        all-MiniLM (384-dim): ~90 MB model
        multi-qa-MiniLM (384-dim): ~90 MB model
        Total embedding model memory: ~1.5 GB
        FAISS index memory: proportional to n_docs * embedding_dim * 4 bytes
        At 2,000 chunks across 3 papers:
            BGE index: 2000 * 1024 * 4 = ~8 MB
            MiniLM index (x2): 2000 * 384 * 4 * 2 = ~6 MB
        Total FAISS: ~14 MB (negligible)
    """

'\n    End-to-end LOTR MergerRetriever RAG pipeline orchestrator.\n\n    Index build strategy:\n        THREE separate FAISS indexes built from the SAME chunks but with\n        DIFFERENT embedding models. Building 3 separate indexes from identical\n        chunks enables embedding-diversity retrieval without requiring 3 separate\n        corpora. Each index "sees" the same documents from a different semantic\n        lens, ensuring that the LOTR merger captures all semantic perspectives.\n\n    Memory considerations:\n        BGE-large (1024-dim): ~1.3 GB model + index memory\n        all-MiniLM (384-dim): ~90 MB model\n        multi-qa-MiniLM (384-dim): ~90 MB model\n        Total embedding model memory: ~1.5 GB\n        FAISS index memory: proportional to n_docs * embedding_dim * 4 bytes\n        At 2,000 chunks across 3 papers:\n            BGE index: 2000 * 1024 * 4 = ~8 MB\n            MiniLM index (x2): 2000 * 384 * 4 * 2 = ~6 MB\n        Total FAISS: ~14 MB (negligible)\n    '

In [40]:
config = LOTRConfig()
logger.info("=== LOTR (MergerRetriever) RAG Pipeline Starting ===")


2026-05-21 20:02:09 | INFO     | lotr_rag | === LOTR (MergerRetriever) RAG Pipeline Starting ===


In [41]:
pdf_paths  = download_pdfs(config)
chunks     = load_and_chunk_documents(pdf_paths, config)


2026-05-21 20:02:12 | INFO     | lotr_rag | Cached: attention_is_all_you_need.pdf  (2163.3 KB)
2026-05-21 20:02:12 | INFO     | lotr_rag | Cached: bert_pretraining_transformers.pdf  (757.0 KB)
2026-05-21 20:02:12 | INFO     | lotr_rag | Cached: rag_knowledge_intensive_nlp.pdf  (864.6 KB)
2026-05-21 20:02:13 | INFO     | lotr_rag |   attention_is_all_you_need.pdf -> 15 pages -> 104 chunks
2026-05-21 20:02:14 | INFO     | lotr_rag |   bert_pretraining_transformers.pdf -> 16 pages -> 173 chunks
2026-05-21 20:02:15 | INFO     | lotr_rag |   rag_knowledge_intensive_nlp.pdf -> 19 pages -> 181 chunks
2026-05-21 20:02:15 | INFO     | lotr_rag | Total chunks: 458


In [42]:
# --- Build 3 separate embedding models ------------------------------
emb_bge  = build_embedding_model(
    config.BGE_MODEL, config.BIENCODER_DEVICE,
    query_instruction=config.BGE_QUERY_INSTRUCTION, normalize=True,
)

2026-05-21 20:02:18 | INFO     | lotr_rag | Loading embedding: BAAI/bge-large-en-v1.5
2026-05-21 20:02:18 | WARNING  | lotr_rag | query_instruction is not supported by this HuggingFaceEmbeddings version; ignoring it.
2026-05-21 20:02:18 | INFO     | sentence_transformers.SentenceTransformer | Load pretrained SentenceTransformer: BAAI/bge-large-en-v1.5
2026-05-21 20:02:18 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/BAAI/bge-large-en-v1.5/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-05-21 20:02:18 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-large-en-v1.5/d4aa6901d3a41ba39fb536a557fa166f842b0e09/modules.json "HTTP/1.1 200 OK"
2026-05-21 20:02:18 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/BAAI/bge-large-en-v1.5/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"


2026-05-21 20:02:18 | WARNING  | huggingface_hub.utils._http | Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-05-21 20:02:18 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-large-en-v1.5/d4aa6901d3a41ba39fb536a557fa166f842b0e09/config_sentence_transformers.json "HTTP/1.1 200 OK"
2026-05-21 20:02:19 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/BAAI/bge-large-en-v1.5/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-05-21 20:02:19 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-large-en-v1.5/d4aa6901d3a41ba39fb536a557fa166f842b0e09/config_sentence_transformers.json "HTTP/1.1 200 OK"
2026-05-21 20:02:19 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/BAAI/bge-large-en-v1.5/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
2

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 5801.87it/s]
BertModel LOAD REPORT from: BAAI/bge-large-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


2026-05-21 20:02:21 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/BAAI/bge-large-en-v1.5/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-05-21 20:02:21 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-large-en-v1.5/d4aa6901d3a41ba39fb536a557fa166f842b0e09/config.json "HTTP/1.1 200 OK"
2026-05-21 20:02:21 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/BAAI/bge-large-en-v1.5/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-05-21 20:02:21 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-large-en-v1.5/d4aa6901d3a41ba39fb536a557fa166f842b0e09/tokenizer_config.json "HTTP/1.1 200 OK"
2026-05-21 20:02:21 | INFO     | httpx | HTTP Request: GET https://huggingface.co/api/models/BAAI/bge-large-en-v1.5/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-05-21 20:02:22 | INFO     | httpx |

In [43]:
emb_mini = build_embedding_model(
        config.MINILM_ALL_MODEL, config.BIENCODER_DEVICE, normalize=True,
    )

2026-05-21 20:02:30 | INFO     | lotr_rag | Loading embedding: sentence-transformers/all-MiniLM-L6-v2
2026-05-21 20:02:30 | INFO     | sentence_transformers.SentenceTransformer | Load pretrained SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2
2026-05-21 20:02:30 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-05-21 20:02:30 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/modules.json "HTTP/1.1 200 OK"
2026-05-21 20:02:31 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-05-21 20:02:31 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2990.29it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


2026-05-21 20:02:34 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-05-21 20:02:34 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/config.json "HTTP/1.1 200 OK"
2026-05-21 20:02:35 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-05-21 20:02:35 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/tokenizer_config.json "HTTP/1.1 200 OK"
2026-05-21 20:02:35 | INFO     | httpx | HTTP Request: GET https://huggingface.co/api/models/sentence-transformers/all-MiniLM-L6-v2/tree/main/additional_chat_templates?recursive=fals

In [44]:
emb_qa   = build_embedding_model(
        config.MINILM_QA_MODEL, config.BIENCODER_DEVICE, normalize=False,
        # multi-qa uses dot-product: normalize=False to preserve magnitude
    )

2026-05-21 20:02:43 | INFO     | lotr_rag | Loading embedding: sentence-transformers/multi-qa-MiniLM-L6-dot-v1
2026-05-21 20:02:43 | INFO     | sentence_transformers.SentenceTransformer | Load pretrained SentenceTransformer: sentence-transformers/multi-qa-MiniLM-L6-dot-v1
2026-05-21 20:02:43 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/sentence-transformers/multi-qa-MiniLM-L6-dot-v1/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-05-21 20:02:44 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/multi-qa-MiniLM-L6-dot-v1/4151c507ffb0f2fcd311cf431f54b5fc7d097851/modules.json "HTTP/1.1 200 OK"
2026-05-21 20:02:44 | INFO     | httpx | HTTP Request: GET https://huggingface.co/api/resolve-cache/models/sentence-transformers/multi-qa-MiniLM-L6-dot-v1/4151c507ffb0f2fcd311cf431f54b5fc7d097851/modules.json "HTTP/1.1 200 OK"
2026-05-21 20:02:44 | INFO     | httpx | HTTP Request: HEAD https://huggingface.c

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6227.58it/s]
BertModel LOAD REPORT from: sentence-transformers/multi-qa-MiniLM-L6-dot-v1
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


2026-05-21 20:03:13 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/sentence-transformers/multi-qa-MiniLM-L6-dot-v1/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-05-21 20:03:13 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/multi-qa-MiniLM-L6-dot-v1/4151c507ffb0f2fcd311cf431f54b5fc7d097851/config.json "HTTP/1.1 200 OK"
2026-05-21 20:03:13 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/sentence-transformers/multi-qa-MiniLM-L6-dot-v1/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-05-21 20:03:14 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/multi-qa-MiniLM-L6-dot-v1/4151c507ffb0f2fcd311cf431f54b5fc7d097851/tokenizer_config.json "HTTP/1.1 200 OK"
2026-05-21 20:03:14 | INFO     | httpx | HTTP Request: GET https://huggingface.co/api/resolve-cache/models/sentence-transformers/multi-qa-MiniLM-L

In [45]:
filter_embeddings = emb_bge   # reuse BGE for all filter operations


In [46]:
# --- Build 3 separate FAISS indexes ---------------------------------
faiss_bge  = build_faiss_index(chunks, emb_bge,  config.FAISS_BGE_DIR)

2026-05-21 20:03:30 | INFO     | lotr_rag | Building FAISS in './faiss_lotr_bge' from 458 chunks ...
2026-05-21 20:06:52 | INFO     | faiss.loader | Loading faiss with AVX512 support.
2026-05-21 20:06:52 | INFO     | faiss.loader | Could not load library with AVX512 support due to:
ModuleNotFoundError("No module named 'faiss.swigfaiss_avx512'")
2026-05-21 20:06:52 | INFO     | faiss.loader | Loading faiss with AVX2 support.
2026-05-21 20:06:52 | INFO     | faiss.loader | Successfully loaded faiss with AVX2 support.
2026-05-21 20:06:52 | INFO     | lotr_rag |   Built. Vectors: 458  (202.57s)


In [47]:
faiss_mini = build_faiss_index(chunks, emb_mini, config.FAISS_MINILM_DIR)


2026-05-21 20:07:16 | INFO     | lotr_rag | Building FAISS in './faiss_lotr_minilm' from 458 chunks ...
2026-05-21 20:07:28 | INFO     | lotr_rag |   Built. Vectors: 458  (11.34s)


In [48]:
faiss_qa   = build_faiss_index(chunks, emb_qa,   config.FAISS_QA_DIR)


2026-05-21 20:07:33 | INFO     | lotr_rag | Building FAISS in './faiss_lotr_qa' from 458 chunks ...
2026-05-21 20:07:44 | INFO     | lotr_rag |   Built. Vectors: 458  (11.17s)


In [55]:
# --- Build BM25 index -----------------------------------------------
bm25 = build_bm25_index(chunks, config)


2026-05-21 20:10:11 | INFO     | lotr_rag | Loading BM25 from 'bm25_lotr\bm25plus.pkl' ...
2026-05-21 20:10:11 | WARNING  | lotr_rag | BM25 cache load failed (Ran out of input); rebuilding ...
2026-05-21 20:10:11 | INFO     | lotr_rag | Building BM25Plus from 458 chunks ...
2026-05-21 20:10:11 | WARNING  | lotr_rag | BM25Plus parameters unsupported by this BM25Retriever version; falling back to BM25Okapi-safe params.


In [56]:
# --- Build LLM -------------------------------------------------------
llm = build_ollama_llm(config)


In [57]:
demo_questions = [
    # Broad semantic query -- embedding diversity expected to improve coverage
    "How does the Transformer model architecture work?",

    # Precise factual -- one retriever may find it; LOTR ensures it is found
    "What BLEU score did the Transformer base model achieve on WMT 2014 EN-DE?",

    # Multi-topic -- clustering should ensure all topics are represented
    "What are the encoder, decoder, and attention components of the Transformer?",

    # QA-style -- multi-qa MiniLM retriever should contribute uniquely here
    "What is the purpose of the [CLS] token in BERT?",

    # Technical synonym test -- BGE and MiniLM may retrieve from different
    # vocabulary perspectives
    "How does self-attention allow the model to relate positions in a sequence?",
]

In [58]:
logger.info("Running %d queries across all 5 LOTR configurations ...", len(demo_questions))

for question in demo_questions:
    run_all_configs(
        question, faiss_bge, faiss_mini, faiss_qa, bm25,
        filter_embeddings, llm, config,
    )

2026-05-21 20:10:49 | INFO     | lotr_rag | Running 5 queries across all 5 LOTR configurations ...

##############################################################################
QUERY: How does the Transformer model architecture work?
##############################################################################

RUNNING: Config1_Baseline
2026-05-21 20:11:04 | INFO     | httpx | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 500 Internal Server Error"
2026-05-21 20:11:04 | ERROR    | lotr_rag | Config Config1_Baseline failed: llama runner process no longer running: 1 CUDA error: shared object initialization failed
CUDA error (status code: 500)
Traceback (most recent call last):
  File "C:\Users\dhanu\AppData\Local\Temp\ipykernel_12632\3506669652.py", line 37, in run_all_configs
    trace = fn(question)
            ^^^^^^^^^^^^
  File "C:\Users\dhanu\AppData\Local\Temp\ipykernel_12632\3506669652.py", line 21, in <lambda>
    "Config1_Baseline": lambda q: run_config1_basel

c:\Users\dhanu\miniconda3\envs\idk\Lib\site-packages\joblib\externals\loky\backend\context.py:131: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] The system cannot find the file specified
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "c:\Users\dhanu\miniconda3\envs\idk\Lib\site-packages\joblib\externals\loky\backend\context.py", line 247, in _count_physical_cores
    cpu_count_physical = _count_physical_cores_win32()
                         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\dhanu\miniconda3\envs\idk\Lib\site-packages\joblib\externals\loky\backend\context.py", line 299, in _count_physical_cores_win32
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "c:\Users\dhanu\miniconda3\envs\idk\Lib\subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^

2026-05-21 20:12:11 | INFO     | lotr_rag | Config4 LOTR+Clustering: 20 merged -> 4 clustered (4 clusters x 1 closest)
2026-05-21 20:12:14 | INFO     | httpx | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"

TRACE: Config4_LOTR_ClusteringFilter
Query: How does the Transformer model architecture work?

  Merged (pre-filter): 20 docs
  After pipeline:      4 docs
  Final to LLM:        4 docs
  LongContextReorder:  not applied

  Retriever contributions:
    FAISS_BGE_large                         : 5 docs
    FAISS_MiniLM_MMR                        : 5 docs
    FAISS_QA_MiniLM                         : 5 docs
    BM25Plus                                : 5 docs

  Final docs:
    [1] Attention Is All You Need p2  (382 chars)
    [2] Bert Pretraining Transformers p2  (419 chars)
    [3] Attention Is All You Need p1  (426 chars)
    [4] Bert Pretraining Transformers p2  (421 chars)

  Latency: retrieval=11188ms  filter=10872ms  gen=16974ms  total=28445ms

  ANSWER:
 

In [59]:
logger.info("=== LOTR MergerRetriever RAG Pipeline Demo Complete ===")


2026-05-21 20:18:07 | INFO     | lotr_rag | === LOTR MergerRetriever RAG Pipeline Demo Complete ===
